In [60]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

TRAIN_PATH = "Train.csv"
TEST_PATH = "Test.csv" if os.path.exists("Test.csv") else "test.csv"
OUTPUT_PATH = "reverted_submission.csv"
TARGET = "is_climate_sensitive"
ID_COLUMN = "ID"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

def make_features(frame):
    data = frame.copy()
    if "deathdate" in data.columns:
        date = pd.to_datetime(data["deathdate"], errors="coerce")
        data["day_of_year"] = date.dt.dayofyear
        data["death_year"] = date.dt.year
        data["death_month"] = date.dt.month
        data["death_quarter"] = date.dt.quarter
        data["temperature_range"] = data["max_temperature"] - data["min_temperature"]
        data["is_rainy_day_current"] = (data["precipitation"] > 0).astype(int)
        data = data.drop(columns=["deathdate"])
    return data.drop(columns=[TARGET, ID_COLUMN], errors="ignore")

X = make_features(train)
y = train[TARGET].astype(int)
X_test = make_features(test)

# Full location strings overfit because almost all test locations are unseen.
X = X.drop(columns=["location"], errors="ignore")
X_test = X_test.drop(columns=["location"], errors="ignore")
categorical = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

transformer = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])
model = Pipeline([
    ("features", transformer),
    ("classifier", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ))
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
model.fit(X_train, y_train)
valid_probability = model.predict_proba(X_valid)[:, 1]
valid_auc = roc_auc_score(y_valid, valid_probability)
thresholds = np.arange(0.10, 0.91, 0.005)

def competition_score(threshold):
    prediction = (valid_probability >= threshold).astype(int)
    return 0.60 * f1_score(y_valid, prediction) + 0.40 * valid_auc

best_threshold = max(thresholds, key=competition_score)
valid_f1 = f1_score(y_valid, (valid_probability >= best_threshold).astype(int))
print(f"Validation F1: {valid_f1:.4f}")
print(f"Validation ROC AUC: {valid_auc:.4f}")
print(f"Validation competition score: {competition_score(best_threshold):.4f}")
print(f"Selected threshold: {best_threshold:.3f}")

model.fit(X, y)
test_probability = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    "TargetF1": (test_probability >= best_threshold).astype(int),
    "TargetRAUC": test_probability
})
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH} with {len(submission)} rows")
submission.head()

C:\Users\boogey\AppData\Local\Temp\ipykernel_13400\3774985215.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = X.select_dtypes(include=["object", "category"]).columns.tolist()


Validation F1: 0.8298
Validation ROC AUC: 0.8219
Validation competition score: 0.8266
Selected threshold: 0.355
Saved reverted_submission.csv with 1030 rows


,ID,TargetF1,TargetRAUC
0,ID_E760D84B,1,0.370596
1,ID_6EDEA907,1,0.603536
2,ID_B9FFC8D8,0,0.323821
3,ID_74C6C94E,0,0.340815
4,ID_0E02825D,1,0.842417
